In [1]:
from collections import Counter
from itertools import pairwise
from collections import defaultdict

from tqdm import tqdm
import pickle
import json
import re

import gc
import copy
import os

In [2]:
from tokenizers import Tokenizer
from transformers import AutoTokenizer

#### Loading the tokenizer

In [3]:
model_dir = f"/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"

print(f"Loading the tokenizer from the following path:\n{model_dir}\n")
tok = AutoTokenizer.from_pretrained(model_dir)


Loading the tokenizer from the following path:
/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1



#### Modifying tokenizer

In [4]:
# The model's tokenizer splits on any ottakshara or virama/halanta
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ', '್ಕ', 'ಾರ', ' ಕನ', '್ನಡ', ' ಭ', 'ಾಷ', 'ೆಯಲ', '್ಲ', 'ಿ']


In [5]:
## Changing the regex pattern in the pre tokenizer, so that kannada words are split correctly
tj = json.loads(tok.backend_tokenizer.to_str())
split = tj["pre_tokenizer"]["pretokenizers"][0]
old = split["pattern"]["Regex"]
new = old.replace(r"[^\r\n\p{L}\p{N}]?\p{L}+", r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+")
assert new != old, "pattern not found; print(old) and adjust"
split["pattern"]["Regex"] = new

In [6]:
# Checking its output
kn_tok = Tokenizer.from_str(json.dumps(tj))
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in kn_tok.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ್ಕಾರ', ' ಕನ್ನಡ', ' ಭಾಷೆಯಲ್ಲಿ']


A note on the characters in the regex patterns:
- \p{L} (Letter): Matches any Unicode letter from any alphabet or language (including ASCII letters).
- \p{N} (Number): Matches any kind of numeric character in Unicode—not just standard ASCII digits (0–9), but also Roman numerals, superscript numbers, and numbers from other scripts (like Arabic-Indic digits). 
- \p{M} (Mark): Matches a mark, which is a character that combines with another character (such as accent marks, diacritics, or vowel signs in scripts like Devanagari or Hebrew) rather than standing alone. 

#### Loading the word frequencies file

In [9]:
merges_path = "/Users/nirajh/Code/Language_adaptation/merges_files/"

with open(merges_path + "word_freq.pkl", "rb") as file:
    word_freq_dict = pickle.load(file)

In [10]:
word_freq_dict.keys()

dict_keys(['word_freq', 'min_freq', 'n_unique_before_filter', 'n_occurrences_before_filter'])

In [11]:
word_freq = word_freq_dict['word_freq']
print(len(word_freq), "unique pre-tokens in the kannada dataset")       # Output: 23277887 unique pre-tokens

5655980 unique pre-tokens in the kannada dataset


In [12]:
print(*word_freq.most_common(10), sep='\n')

('.', 60116249)
(',', 41402116)
('Ġ', 21435058)
('.Ċ', 18937315)
('Ġà²Ī', 9952450)
('Ġà²®à²¤à³įà²¤à³ģ', 9594566)
('Ġà²İà²Ĥà²¦à³ģ', 7650822)
('Ġ(', 4349045)
(')', 3621964)
('Ġà²ħà²µà²°à³ģ', 3535965)


In [13]:
# Getting the model's pretokens for each word (display word); this applies merges that the model has already learned
tok_model = kn_tok.model
word_tokens = {w: [t.value for t in tok_model.tokenize(w)] for w in tqdm(word_freq)}

100%|██████████| 5655980/5655980 [00:38<00:00, 145856.04it/s]


In [14]:
print("Printing a few of these word-value pairs:")

ind = 0
for w, v in word_tokens.items():
    print(f"{w}, {v}")
    ind += 1
    if ind>=10:
        break

Printing a few of these word-value pairs:
à²¨à²Ĺà²°à²¦, ['à²', '¨', 'à²', 'Ĺ', 'à²', '°', 'à²', '¦']
Ġà²ķà³ģà²®à²¾à²°à²¸à³įà²µà²¾à²®à²¿, ['Ġà²', 'ķ', 'à³', 'ģ', 'à²', '®', 'à²', '¾', 'à²', '°', 'à²', '¸', 'à³', 'į', 'à²', 'µ', 'à²', '¾', 'à²', '®', 'à²', '¿']
Ġà²¦à³ĩà²µà²¸à³įà²¥à²¾à²¨à²¦, ['Ġà²', '¦', 'à³', 'ĩ', 'à²', 'µ', 'à²', '¸', 'à³', 'į', 'à²', '¥', 'à²', '¾', 'à²', '¨', 'à²', '¦']
Ġà²®à³ģà²ĸà³įà²¯à²°à²¸à³įà²¤à³Ĩ, ['Ġà²', '®', 'à³', 'ģ', 'à²', 'ĸ', 'à³', 'į', 'à²', '¯', 'à²', '°', 'à²', '¸', 'à³', 'į', 'à²', '¤', 'à³', 'Ĩ']
Ġà²¬à²³à²¿, ['Ġà²', '¬', 'à²', '³', 'à²', '¿']
Ġà²ĩà²°à³ģà²µ, ['Ġà²', 'ĩ', 'à²', '°', 'à³', 'ģ', 'à²', 'µ']
Ġà²ķà³ģà²°à³ģà²¬à²°, ['Ġà²', 'ķ', 'à³', 'ģ', 'à²', '°', 'à³', 'ģ', 'à²', '¬', 'à²', '°']
Ġà²¸à²Ĥà²ĺà²¦, ['Ġà²', '¸', 'à²', 'Ĥ', 'à²', 'ĺ', 'à²', '¦']
Ġà²Ĩà²µà²°à²£à²¦à²²à³įà²²à²¿, ['Ġà²', 'Ĩ', 'à²', 'µ', 'à²', '°', 'à²', '£', 'à²', '¦', 'à²', '²', 'à³', 'į', 'à²', '²', 'à²', '¿']
Ġà²µà²¸à²¤à²¿, ['Ġà²', 'µ', 'à²', '¸', 'à²', '¤', 'à²', '¿']


#### Checking what these Kannada words are

To do that, convert the byte-level stand-in characters back to raw bytes, then decode the bytes as UTF-8.

In [15]:
from functools import lru_cache

@lru_cache()
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
            #print(f"This b is not in bs: {b}; it's char is: '{chr(b)}'")
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

In [16]:
byte_decoder = {v: k for k, v in bytes_to_unicode().items()}        # This inverts the byte encoder

def to_bytes(s):
    return bytes(byte_decoder[c] for c in s)         # 's' is a byte-level token string like 'Ġà²ķ'. byte_decoder() returns the byte value from the lookup table

def to_text(s):
    return to_bytes(s).decode("utf-8", errors="replace")        # Interprets the recovered bytes as UTF-8 to get real text back


In [17]:
for w, c in word_freq.most_common(20):
    print(f"{to_text(w)!r:25} {c:>12,}")

'.'                         60,116,249
','                         41,402,116
' '                         21,435,058
'.\n'                       18,937,315
' ಈ'                         9,952,450
' ಮತ್ತು'                     9,594,566
' ಎಂದು'                      7,650,822
' ('                         4,349,045
')'                          3,621,964
' ಅವರು'                      3,535,965
' ಹಾಗೂ'                      3,290,607
' ಒಂದು'                      3,233,622
" '"                         3,138,645
'\n'                         3,071,853
'-'                          2,794,055
' ಆದರೆ'                      2,758,565
"'"                          2,722,745
' ಅವರ'                       2,497,065
' -'                         2,476,254
' ಬಗ್ಗೆ'                     2,289,147


In [18]:
bytes([32, 224, 136]).decode("utf-8", errors="replace")

' ��'

#### Removing pretokens that are not Kannada words

In [19]:
KANNADA = re.compile(r"[\u0C80-\u0CFF]")

kn_word_freq = Counter({w: c for w, c in tqdm(word_freq.items()) if KANNADA.search(to_text(w))})

print(f"Pre-tokens kept: {len(kn_word_freq):,} of {len(word_freq):,}")
print(f"Occurrences kept: {sum(kn_word_freq.values()) / sum(word_freq.values()):.2%}")

for w, c in kn_word_freq.most_common(10):
    print(f"{to_text(w)!r:25} {c:>12,}")

100%|██████████| 5655980/5655980 [00:06<00:00, 871551.27it/s]


Pre-tokens kept: 5,029,378 of 5,655,980
Occurrences kept: 77.23%
' ಈ'                         9,952,450
' ಮತ್ತು'                     9,594,566
' ಎಂದು'                      7,650,822
' ಅವರು'                      3,535,965
' ಹಾಗೂ'                      3,290,607
' ಒಂದು'                      3,233,622
' ಆದರೆ'                      2,758,565
' ಅವರ'                       2,497,065
' ಬಗ್ಗೆ'                     2,289,147
' ಮೇಲೆ'                      2,280,629


In [20]:
# Checking the pretokenized display characters and their corresponding kannada characters
for w, v in kn_word_freq.most_common(10):
    print(f'''{word_tokens[w]!r:70} {to_text(w)}''')


['Ġà²', 'Ī']                                                            ಈ
['Ġà²', '®', 'à²', '¤', 'à³', 'į', 'à²', '¤', 'à³', 'ģ']                ಮತ್ತು
['Ġà²', 'İ', 'à²', 'Ĥ', 'à²', '¦', 'à³', 'ģ']                           ಎಂದು
['Ġà²', 'ħ', 'à²', 'µ', 'à²', '°', 'à³', 'ģ']                           ಅವರು
['Ġà²', '¹', 'à²', '¾', 'à²', 'Ĺ', 'à³', 'Ĥ']                           ಹಾಗೂ
['Ġà²', 'Ĵ', 'à²', 'Ĥ', 'à²', '¦', 'à³', 'ģ']                           ಒಂದು
['Ġà²', 'Ĩ', 'à²', '¦', 'à²', '°', 'à³', 'Ĩ']                           ಆದರೆ
['Ġà²', 'ħ', 'à²', 'µ', 'à²', '°']                                      ಅವರ
['Ġà²', '¬', 'à²', 'Ĺ', 'à³', 'į', 'à²', 'Ĺ', 'à³', 'Ĩ']                ಬಗ್ಗೆ
['Ġà²', '®', 'à³', 'ĩ', 'à²', '²', 'à³', 'Ĩ']                           ಮೇಲೆ


In [21]:
kn_word_freq.most_common(10)

[('Ġà²Ī', 9952450),
 ('Ġà²®à²¤à³įà²¤à³ģ', 9594566),
 ('Ġà²İà²Ĥà²¦à³ģ', 7650822),
 ('Ġà²ħà²µà²°à³ģ', 3535965),
 ('Ġà²¹à²¾à²Ĺà³Ĥ', 3290607),
 ('Ġà²Ĵà²Ĥà²¦à³ģ', 3233622),
 ('Ġà²Ĩà²¦à²°à³Ĩ', 2758565),
 ('Ġà²ħà²µà²°', 2497065),
 ('Ġà²¬à²Ĺà³įà²Ĺà³Ĩ', 2289147),
 ('Ġà²®à³ĩà²²à³Ĩ', 2280629)]

### Merging the most frequent token pairs

In [22]:
def create_pair_counts(word_freq, word_tok:dict):

    # A counter to tally the number of times a pair ocurrs
    pair_counts = Counter()

    # Creating a list of the words in which each pair occurs
    pairs_to_words = defaultdict(list)

    for idx, (display_word, count) in enumerate(tqdm(word_freq.items())):
        
        dc_list = word_tok[display_word]
        local_pairs = Counter(pairwise(dc_list))
        weighted_pairs = Counter({pair: c * count for pair, c in local_pairs.items()})
        pair_counts.update(weighted_pairs)

        # Updating the words in which a pair is seen
        for p in local_pairs.keys():
            pairs_to_words[p].append(display_word)

    return pair_counts, pairs_to_words


In [23]:
pair_counts_op, pairs_to_words_op = create_pair_counts(kn_word_freq, word_tokens)

100%|██████████| 5029378/5029378 [00:26<00:00, 188704.82it/s]


In [24]:
pair_counts_op.most_common(5)

[(('à³', 'į'), 574565151),
 (('į', 'à²'), 509182313),
 (('à²', '¿'), 471364707),
 (('¿', 'à²'), 360465629),
 (('à²', '°'), 338308884)]

In [24]:
def merge(tokens_, top_):

    new_toks = []
    i = 0
    merged_tokens = ''.join(top_)

    while i < len(tokens_):
        if (i<(len(tokens_)-1)) and (tokens_[i]==top_[0]) and (tokens_[i+1]==top_[1]):
            new_toks.append(merged_tokens)
            i += 2
        else:
            new_toks.append(tokens_[i])
            i += 1

    return new_toks

In [25]:
def create_merges(pair_counts_:Counter, word_freq_:Counter, word_toks_:dict, pairs_to_words_:defaultdict, check_flag:bool = False):

    # Getting the top pair
    top_pair = max(pair_counts_, key=lambda k: pair_counts_[k])

    # Getting a count of the number of times the top pair occurs
    merge_count = pair_counts_[top_pair]

    #for w in pairs_to_words_[top_pair]:
    for w in dict.fromkeys(pairs_to_words_[top_pair]):   # unique words, original order, a copy

        # Looking up the tokens in this word
        toks = word_toks_[w]

        # Counting the number of times each token pair occurs in the current word
        old_pairs = Counter(pairwise(toks))

        for p, c in old_pairs.items():
            pair_counts_[p] -= c * word_freq_[w]        # Multiplying frequency by count as a pair may occur more than once in the current word

            if pair_counts_[p] == 0:
                del pair_counts_[p]                 # Removing this pair from the Counter as its count is zero; This is faster than using 'pair_counts_ = +pair_counts_' outside this for loop, which copies the entire counter
                pairs_to_words_.pop(p, None)        # The current pair p is no longer present in any word, so its word list is stale

            elif check_flag and pair_counts_[p] < 0:
                raise AssertionError(f"Negative count for {p}: {pair_counts_[p]}")

        # Updating the tokens list for the current word directly in the dictionary
        word_toks_[w] = merge(toks, top_pair)

        for p in pairwise(word_toks_[w]):
            pair_counts_[p] += word_freq_[w]        # Adding the pair count for the new token pairs in this word; no need to multiply here as this loop is iterating over a list and not a counter

            if (p not in old_pairs) or (p not in pairs_to_words_):          # Only adding pairs that are newly formed (checking for that in two different ways)
                pairs_to_words_[p].append(w)                           # This word has one or more different pairs now; therefore, adding the word to those pairs' lists
        
        # By using pairwise() and the above two loops, we avoid deleting items based on their index (since the index shifts on deleting and it's hard to keep track for multiple occurrences of a pair in the same word)
        # The first pairwise for loop could be simpler, but that would make the second more more complex; this implementation keeps both fairly simple

    # Also removing the top pair from the pairs to words mapping
    pairs_to_words_.pop(top_pair, None)

    if check_flag:
        # Checking if the pair counts and the list of pairs match (if they do, it means the pair counts don't have to be rebuilt in every iteration of this for loop)
        new_pair_counts, _ = create_pair_counts(word_freq_, word_toks_)
        if new_pair_counts!= pair_counts_:
            print(f"Old pair counts: {len(pair_counts_)} \nNew pair counts: {len(new_pair_counts)}")
            raise AssertionError
    
    return top_pair, merge_count, pair_counts_, word_toks_, pairs_to_words_


In [26]:
pair_counts_wip = copy.deepcopy(pair_counts_op)
pairs_to_words = copy.deepcopy(pairs_to_words_op)

In [27]:
num_merges = 100
merges = []

for idx in tqdm(range(num_merges), desc="Merging token pairs"):

    merged_pair, merged_count, pair_counts_wip, word_tokens, pairs_to_words = create_merges(
        pair_counts_=pair_counts_wip, 
        word_freq_=kn_word_freq, 
        word_toks_=word_tokens,
        pairs_to_words_=pairs_to_words,
        check_flag=True if idx<=2 else False
    )

    merges.append((merged_pair, merged_count))


Merging token pairs: 100%|██████████| 100/100 [20:21<00:00, 12.22s/it]


In [ ]:
output_path = "/Users/nirajh/Code/Language_adaptation/merges_files/"
os.makedirs(output_path, exist_ok=True)

final_path = os.path.join(output_path, "new_merges.pkl")
tmp_path = final_path + ".tmp"


with open(tmp_path, "wb") as f:
    pickle.dump(merges, f, protocol=pickle.HIGHEST_PROTOCOL)
    f.flush()
    os.fsync(f.fileno())


print("Wrote tmp file to disk")

os.replace(tmp_path, final_path)

print("Replaced tmp file with the final file")

Wrote tmp file to disk
Replaced tmp file with the final file


In [29]:
# verify by reading it back
with open(final_path, "rb") as f:
    check = pickle.load(f)

assert len(check) == len(merges)

print(f"Wrote and verified {final_path} ({os.path.getsize(final_path) / 1e6:.1f} MB)")

Wrote and verified /Users/nirajh/Code/Language_adaptation/merges_work/new_merges.pkl (0.0 MB)


In [25]:
import os
import pickle

In [26]:
output_path = "/Users/nirajh/Code/Language_adaptation/merges_files/"
os.makedirs(output_path, exist_ok=True)

final_path = os.path.join(output_path, "new_merges.pkl")

In [27]:
final_path

'/Users/nirajh/Code/Language_adaptation/merges_files/new_merges.pkl'

In [28]:
# verify by reading it back
with open(final_path, "rb") as f:
    check = pickle.load(f)

In [29]:
check[:5]

[(('à³', 'į'), 574565151),
 (('à³į', 'à²'), 509178184),
 (('à²', '¿'), 471344212),
 (('à²¿', 'à²'), 360436006),
 (('à³', 'ģ'), 323417073)]

In [30]:
check[-5:]

[(('Ġà²«à³Ģà²ļ', 'à²°à³įà²¸'), 15725),
 (('Ġà²¸à²Ĥà²Ĺ', 'à²¤à²¿à²Ĺà²³'), 15724),
 (('Ġà²¸à³įà²Ł', 'à³Īà²²à³į'), 15722),
 (('Ġà²İà²¸', 'à²¿à²¬à²¿'), 15722),
 (('Ġà²¸à³ĩà²°', 'à²¿à²¦à²°à³ģ'), 15721)]

In [31]:
len(check)

16000

In [32]:
for (a, b), c in check[:25]:
    print(f"{to_text(a)!r:12} + {to_text(b)!r:12} -> {to_text(a + b)!r:16} {c:>12,}")

'�'          + '�'          -> '್'               574,565,151
'್'          + '�'          -> '್�'              509,178,184
'�'          + '�'          -> 'ಿ'               471,344,212
'ಿ'          + '�'          -> 'ಿ�'              360,436,006
'�'          + '�'          -> 'ು'               323,417,073
'�'          + '�'          -> 'ಾ'               309,998,325
'ಾ'          + '�'          -> 'ಾ�'              289,534,577
'�'          + '�'          -> 'ೆ'               244,651,272
'�'          + '�'          -> 'ರ'               194,781,000
'�'          + '�'          -> 'ದ'               184,217,528
'�'          + '�'          -> 'ವ'               124,769,418
'�'          + '�'          -> 'ಲ'               122,715,220
'�'          + '�'          -> 'ನ'               120,718,556
'�'          + '�'          -> 'ಂ'               119,746,091
'�'          + '�'          -> 'ತ'               112,127,602
'�'          + '�'          -> 'ಗ'                94,852,259
'�'          + '�'      

In [35]:
for (a, b), c in check[500:525]:
    print(f"{to_text(a)!r:12} + {to_text(b)!r:12} -> {to_text(a + b)!r:16} {c:>12,}")

'ಿತ'         + 'ಿ'          -> 'ಿತಿ'               1,515,537
'ೋ'          + 'ಷ'          -> 'ೋಷ'                1,513,909
'ಿಸಿದ'       + 'ರು'         -> 'ಿಸಿದರು'            1,505,654
'ಾನ'         + '್'          -> 'ಾನ್'               1,503,017
'ನ'          + 'ಾ'          -> 'ನಾ'                1,501,806
' ಪ'         + 'ೊ'          -> ' ಪೊ'               1,493,843
' ಕೆಲ'       + 'ಸ'          -> ' ಕೆಲಸ'             1,491,702
'ಿಕ'         + '್ಕ'         -> 'ಿಕ್ಕ'              1,486,290
' ಶ'         + '್ರೀ'        -> ' ಶ್ರೀ'             1,483,900
' ನ'         + 'ನ್ನ'        -> ' ನನ್ನ'             1,483,000
' ಹ'         + 'ಾಕ'         -> ' ಹಾಕ'              1,482,273
'ುತ್ತ'       + 'ಿದೆ'        -> 'ುತ್ತಿದೆ'           1,474,004
' ಮು'        + 'ಖ್ಯ'        -> ' ಮುಖ್ಯ'            1,471,193
'ುದ'         + 'ೇ'          -> 'ುದೇ'               1,469,027
'್ನ'         + 'ಲ್ಲಿ'       -> '್ನಲ್ಲಿ'            1,461,664
'ೀ'          + 'ಗ'          -> 'ೀಗ'                1,460,137
'ಿಯ'         + 'ಾಗ'     

In [ ]:
# Viewing bytes instead of characters

for (a, b), c in check[:20]:
    print(f"{to_bytes(a).hex(' '):10} + {to_bytes(b).hex(' '):10} -> {to_text(a + b)!r:16} {c:>12,}")

e0 b3      + 8d         -> '್'               574,565,151
e0 b3 8d   + e0 b2      -> '್�'              509,178,184
e0 b2      + bf         -> 'ಿ'               471,344,212
e0 b2 bf   + e0 b2      -> 'ಿ�'              360,436,006
e0 b3      + 81         -> 'ು'               323,417,073
e0 b2      + be         -> 'ಾ'               309,998,325
e0 b2 be   + e0 b2      -> 'ಾ�'              289,534,577
e0 b3      + 86         -> 'ೆ'               244,651,272
e0 b2      + b0         -> 'ರ'               194,781,000
e0 b2      + a6         -> 'ದ'               184,217,528
e0 b2      + b5         -> 'ವ'               124,769,418
e0 b2      + b2         -> 'ಲ'               122,715,220
e0 b2      + a8         -> 'ನ'               120,718,556
e0 b2      + 82         -> 'ಂ'               119,746,091
e0 b2      + a4         -> 'ತ'               112,127,602
e0 b2      + 97         -> 'ಗ'                94,852,259
e0 b2      + 95         -> 'ಕ'                85,290,093
e0 b2      + b3         -> 'ಳ' 

In [ ]:
# Checking the number of partial-character tokens ('\ufffd' or �)

partial = [((a, b), c) for (a, b), c in check if "\ufffd" in to_text(a + b)]
print(f"{len(partial)} of {len(check)} merges produce partial-character tokens")
print(f"In the first 2,000: {sum(1 for (a, b), _ in check[:2000] if chr(0xFFFD) in to_text(a + b))}")

13 of 16000 merges produce partial-character tokens
In the first 2,000: 11
